# QNLP Fake/Real News Classifier — Production-Oriented Pipeline

**Honest framing, read this first:** this is a *purely* QNLP model — every trainable
parameter lives inside quantum circuits, there is no classical classifier hidden anywhere
(document-level aggregation below is a parameter-free mean, not a learned layer). That
purity comes with a hard, physical constraint: classical simulation of a quantum circuit
costs memory that grows as roughly 2^(qubit count), and here qubit count tracks word
count. There is no configuration of "purely QNLP simulated on classical hardware" that
accepts unbounded document length — that isn't a code limitation, it's the same reason no
one runs 300-qubit simulations anywhere, on any hardware, today.

What *is* achievable, and what this notebook aims for: a properly engineered pipeline
around a **bounded, justified** QNLP model — with the software qualities a production
system needs (structured logging, checkpointing, evaluation metrics, model persistence,
an inference function, and a serving-layer example), rather than an unbounded one that
can't finish.

**Fixes vs. earlier iterations of this notebook** (all bugs found during development):
- Sentence-chunking caps (`MAX_WORDS_PER_SENTENCE`, `MAX_SENTENCES_PER_DOC`) are read live
  from config, not baked in as stale function defaults.
- Exactly one definition of `split_sentences` / `doc_to_sentence_chunks` (earlier versions
  accumulated duplicate/conflicting definitions across edits).
- Config values are validated immediately (`assert ... > 0`) at the point of definition,
  not discovered via a confusing downstream stack trace.
- Sentence splitting is fully offline (regex-based) — no `nltk` download, which isn't
  reachable in this environment.
- Added regression tests on the chunking logic so a bad config value (like the `-1` that
  crept in earlier) fails loudly and immediately instead of silently corrupting data.


## 1. Setup, logging, and configuration

In [1]:
!pip install -q -U pennylane scikit-learn scipy lambeq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.4 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 7.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 11.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.6/269.6 kB 4.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 13.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.3/364.3 kB 8.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 1

In [2]:
import torch
import random
import numpy as np
import logging

# --- reproducibility ---
SEED = 12
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# --- structured logging instead of scattered prints ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
)
logger = logging.getLogger("qnlp_fake_news")


In [5]:
# --------------------------------------------------------------
# CONFIG -- calibrated for stable, feasible QNLP simulation on Kaggle
# --------------------------------------------------------------

# Quantum circuit / ansatz
N_LAYERS = 8                    # 1-3 layers is standard for IQP in QNLP literature (prevents barren plateaus & OOM)
N_SINGLE_QUBIT_PARAMS = 12       # Single-qubit rotation parameters per word box

# Sentence chunking -- bounds the qubit count of any single circuit
MAX_WORDS_PER_SENTENCE = 12     # Keeps statevector simulation fast (~2^12 state space)
MAX_SENTENCES_PER_DOC = 5       # 3-5 sentences captures key news leads without exponential simulation cost

# Dataset size -- calibrated for pure QNLP simulation limits on Kaggle (9-hour limit)
# Note: For full datasets (20k+ docs), use Track A (QHBERT: DistilBERT + VQC). Pure QNLP (DisCoCat) uses subsets:
N_TRAIN = 300                   # Prototyping benchmark size (scale to 500-1000 after timing first epoch)
N_DEV = 60
N_TEST = 60

# Training
DOC_BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.05
EVAL_EVERY = 1

MODEL_CHECKPOINT_PATH = "qnlp_fake_news_best.lt"
MODEL_FINAL_PATH = "qnlp_fake_news_final.lt"

# --- fail fast on bad config, at the point of definition ---
assert N_LAYERS > 0, f"N_LAYERS must be positive, got {N_LAYERS}"
assert N_SINGLE_QUBIT_PARAMS > 0
assert MAX_WORDS_PER_SENTENCE > 0, f"MAX_WORDS_PER_SENTENCE must be positive, got {MAX_WORDS_PER_SENTENCE}"
assert MAX_SENTENCES_PER_DOC > 0, f"MAX_SENTENCES_PER_DOC must be positive, got {MAX_SENTENCES_PER_DOC}"
assert N_TRAIN > 0 and N_DEV > 0 and N_TEST > 0
assert DOC_BATCH_SIZE > 0 and EPOCHS > 0 and LEARNING_RATE > 0

logger.info(
    "Config OK: N_LAYERS=%d MAX_WORDS_PER_SENTENCE=%d MAX_SENTENCES_PER_DOC=%d "
    "N_TRAIN=%d N_DEV=%d N_TEST=%d",
    N_LAYERS, MAX_WORDS_PER_SENTENCE, MAX_SENTENCES_PER_DOC, N_TRAIN, N_DEV, N_TEST,
)

2026-08-26 19:34:47,861 [INFO] Config OK: N_LAYERS=8 MAX_WORDS_PER_SENTENCE=12 MAX_SENTENCES_PER_DOC=5 N_TRAIN=300 N_DEV=60 N_TEST=60


## 2. Load and clean the dataset

In [6]:
import glob
import pandas as pd

def _find_csv(name):
    matches = glob.glob(f"{name}", recursive=True)
    return matches[0] if matches else f"/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/{name}"


def load_fake_real_news():
    fake = pd.read_csv(_find_csv("Fake.csv"))
    real = pd.read_csv(_find_csv("True.csv"))
    fake["label"] = 0   # Fake news
    real["label"] = 1   # Real news
    df = pd.concat([fake, real], ignore_index=True)
    df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")
    return df[["content", "label"]]


df_final = load_fake_real_news()
logger.info("Raw dataset shape: %s", df_final.shape)
logger.info("Label distribution:\n%s", df_final["label"].value_counts().to_string())
df_final.head()


2026-08-26 19:34:56,503 [INFO] Raw dataset shape: (44898, 2)
2026-08-26 19:34:56,506 [INFO] Label distribution:
label
0    23481
1    21417


,content,label
0,Donald Trump Sends Out Embarrassing New Year’...,0
1,Drunk Bragging Trump Staffer Started Russian ...,0
2,Sheriff David Clarke Becomes An Internet Joke...,0
3,Trump Is So Obsessed He Even Has Obama’s Name...,0
4,Pope Francis Just Called Out Donald Trump Dur...,0


In [7]:
df = df_final.copy()
df["content"] = df["content"].fillna("").astype(str).str.strip()
df = df[df["content"] != ""]
df = df.drop_duplicates(subset=["content"])
df["label"] = df["label"].astype(int)
logger.info("Cleaned dataset shape: %s", df.shape)


2026-08-26 19:34:58,025 [INFO] Cleaned dataset shape: (39103, 2)


In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

logger.info("Train: %s | Validation: %s | Test: %s", train_df.shape, val_df.shape, test_df.shape)

train_texts, train_labels = train_df["content"].tolist(), train_df["label"].tolist()
val_texts, val_labels = val_df["content"].tolist(), val_df["label"].tolist()
test_texts, test_labels = test_df["content"].tolist(), test_df["label"].tolist()


2026-08-26 19:35:00,016 [INFO] Train: (27372, 2) | Validation: (5865, 2) | Test: (5866, 2)


## 3. Sentence chunking (single definition, offline, config-validated)

This is the piece that broke earlier via stale defaults and duplicate definitions. Here
there is exactly one definition of each function, caps are read live from the config cell
above, and assertions confirm the invariants actually hold on every call.


In [9]:
import re

_SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text):
    """Offline, dependency-free sentence splitter. Not perfect around
    abbreviations (e.g. 'U.S.') but doesn't need to be -- we just need
    real, in-order word chunks, not linguistically perfect boundaries."""
    text = str(text).strip()
    if not text:
        return []
    parts = _SENTENCE_SPLIT_RE.split(text)
    return [p.strip() for p in parts if p.strip()]


def doc_to_sentence_chunks(text):
    """Split a document into up to MAX_SENTENCES_PER_DOC sentences, each
    capped at MAX_WORDS_PER_SENTENCE words. Reads the caps LIVE from the
    globals set in the config cell -- re-running just that cell is enough
    to change behavior, no stale bindings."""
    sentences = split_sentences(text)[:MAX_SENTENCES_PER_DOC]
    chunks = []
    for s in sentences:
        words = s.split()[:MAX_WORDS_PER_SENTENCE]
        if words:
            chunks.append(" ".join(words))

    assert len(chunks) <= MAX_SENTENCES_PER_DOC, (
        f"got {len(chunks)} chunks but MAX_SENTENCES_PER_DOC={MAX_SENTENCES_PER_DOC}"
    )
    max_words_seen = max((len(c.split()) for c in chunks), default=0)
    assert max_words_seen <= MAX_WORDS_PER_SENTENCE, (
        f"got a chunk with {max_words_seen} words but MAX_WORDS_PER_SENTENCE={MAX_WORDS_PER_SENTENCE}"
    )
    return chunks


def build_doc_chunks(texts, labels, n_docs):
    doc_sentence_lists, doc_labels = [], []
    for text, label in zip(texts, labels):
        chunks = doc_to_sentence_chunks(text)
        if chunks:
            doc_sentence_lists.append(chunks)
            doc_labels.append(label)
        if len(doc_sentence_lists) >= n_docs:
            break
    return doc_sentence_lists, doc_labels


### 3a. Regression tests

These would have caught the earlier `-1` config bug immediately, at the source, instead of
several confusing steps downstream. Run this cell every time the config or chunking logic
changes.


In [10]:
def _run_chunking_tests():
    # sanity: caps are respected on a document that clearly exceeds them
    long_doc = " ".join([f"This is sentence number {i} with some extra words here." for i in range(100)])
    chunks = doc_to_sentence_chunks(long_doc)
    assert len(chunks) <= MAX_SENTENCES_PER_DOC, "sentence cap violated"
    assert all(len(c.split()) <= MAX_WORDS_PER_SENTENCE for c in chunks), "word cap violated"

    # sanity: empty / whitespace-only text produces no chunks, not an error
    assert doc_to_sentence_chunks("") == []
    assert doc_to_sentence_chunks("   ") == []

    # sanity: a short document is NOT padded or truncated below its real length
    short_doc = "Reuters reported the news today. Officials declined to comment."
    chunks = doc_to_sentence_chunks(short_doc)
    assert len(chunks) == 2, f"expected 2 sentences, got {len(chunks)}"

    # sanity: config values themselves are positive (redundant with config cell,
    # but this test should fail loudly even if someone edits config without re-running it)
    assert MAX_SENTENCES_PER_DOC > 0, "MAX_SENTENCES_PER_DOC must be positive -- do not use -1 as an 'unlimited' sentinel, Python slicing treats negative numbers as 'from the end', not 'no limit'"
    assert MAX_WORDS_PER_SENTENCE > 0

    logger.info("All chunking regression tests passed.")

_run_chunking_tests()


2026-08-26 19:35:00,063 [INFO] All chunking regression tests passed.


In [11]:
train_doc_sentences, train_doc_labels = build_doc_chunks(train_texts, train_labels, N_TRAIN)
dev_doc_sentences, dev_doc_labels = build_doc_chunks(val_texts, val_labels, N_DEV)
test_doc_sentences, test_doc_labels = build_doc_chunks(test_texts, test_labels, N_TEST)

for name, docs in [("train", train_doc_sentences), ("dev", dev_doc_sentences), ("test", test_doc_sentences)]:
    n_sent = [len(d) for d in docs]
    n_words = [len(s.split()) for d in docs for s in d]
    logger.info(
        "%s: %d docs | sentences/doc avg=%.1f max=%d | words/sentence avg=%.1f max=%d",
        name, len(docs), np.mean(n_sent), max(n_sent), np.mean(n_words), max(n_words),
    )


2026-08-26 19:35:00,131 [INFO] train: 300 docs | sentences/doc avg=4.8 max=5 | words/sentence avg=10.8 max=12
2026-08-26 19:35:00,132 [INFO] dev: 60 docs | sentences/doc avg=4.7 max=5 | words/sentence avg=11.0 max=12
2026-08-26 19:35:00,133 [INFO] test: 60 docs | sentences/doc avg=4.6 max=5 | words/sentence avg=10.8 max=12


## 4. Install dependencies

In [12]:
import sys, lambeq
logger.info("Python: %s", sys.version.split()[0])
logger.info("lambeq: %s", lambeq.__version__)


2026-08-26 19:35:27,548 [INFO] Python: 3.12.13
2026-08-26 19:35:27,549 [INFO] lambeq: 0.5.0


## 5. Parser setup

`BobcatParser` (lambeq's real syntax-aware parser) still requires a download from a dead
host (`qnlp.cambridgequantum.com`) -- confirmed via lambeq's own open GitHub issue and no
bundled/mirrored alternative as of this writing. `spiders_reader` is used instead: it's a
bag-of-words-style reader (no real grammar), fully offline. If `BobcatParser` becomes
usable again, swapping it in here is the one change needed for syntax-aware diagrams.


In [13]:
from lambeq import spiders_reader
parser = spiders_reader


## 6. Deduplicate, parse, and build circuits

In [14]:
def flatten_with_dedup(doc_sentence_lists):
    """News articles share a lot of boilerplate; parse each UNIQUE sentence
    chunk once and let documents reference it by index."""
    sentence_to_idx, unique_sentences, doc_index_lists = {}, [], []
    for doc in doc_sentence_lists:
        idxs = []
        for s in doc:
            if s not in sentence_to_idx:
                sentence_to_idx[s] = len(unique_sentences)
                unique_sentences.append(s)
            idxs.append(sentence_to_idx[s])
        doc_index_lists.append(idxs)
    return unique_sentences, doc_index_lists


def parse_unique_sentences(unique_sents):
    """Parse each unique sentence chunk individually, dropping any that fail
    rather than aborting the whole run."""
    diagrams = [None] * len(unique_sents)
    ok_mask = [True] * len(unique_sents)
    for i, s in enumerate(unique_sents):
        try:
            diagrams[i] = parser.sentence2diagram(s)
        except Exception as e:
            ok_mask[i] = False
            logger.debug("Failed to parse sentence %d (%r): %s", i, s, e)
    n_failed = ok_mask.count(False)
    if n_failed:
        logger.warning("%d/%d sentence chunks failed to parse and will be dropped", n_failed, len(unique_sents))
    return diagrams, ok_mask


def remap_after_drop(diagrams, ok_mask, doc_idxs):
    new_diagrams, old_to_new = [], {}
    for old_i, ok in enumerate(ok_mask):
        if ok:
            old_to_new[old_i] = len(new_diagrams)
            new_diagrams.append(diagrams[old_i])
    new_doc_idxs = [[old_to_new[i] for i in doc if i in old_to_new] for doc in doc_idxs]
    return new_diagrams, new_doc_idxs


def drop_empty_docs(doc_idxs, doc_labels):
    keep = [(idxs, lab) for idxs, lab in zip(doc_idxs, doc_labels) if idxs]
    if not keep:
        return [], []
    idxs_out, labels_out = zip(*keep)
    return list(idxs_out), list(labels_out)


train_unique_sents, train_doc_idxs = flatten_with_dedup(train_doc_sentences)
dev_unique_sents, dev_doc_idxs = flatten_with_dedup(dev_doc_sentences)
test_unique_sents, test_doc_idxs = flatten_with_dedup(test_doc_sentences)
logger.info("Unique sentence chunks -> train: %d dev: %d test: %d",
            len(train_unique_sents), len(dev_unique_sents), len(test_unique_sents))

train_raw_diagrams, train_ok = parse_unique_sentences(train_unique_sents)
dev_raw_diagrams, dev_ok = parse_unique_sentences(dev_unique_sents)
test_raw_diagrams, test_ok = parse_unique_sentences(test_unique_sents)

train_diagrams, train_doc_idxs = remap_after_drop(train_raw_diagrams, train_ok, train_doc_idxs)
dev_diagrams, dev_doc_idxs = remap_after_drop(dev_raw_diagrams, dev_ok, dev_doc_idxs)
test_diagrams, test_doc_idxs = remap_after_drop(test_raw_diagrams, test_ok, test_doc_idxs)

train_doc_idxs, train_doc_labels = drop_empty_docs(train_doc_idxs, train_doc_labels)
dev_doc_idxs, dev_doc_labels = drop_empty_docs(dev_doc_idxs, dev_doc_labels)
test_doc_idxs, test_doc_labels = drop_empty_docs(test_doc_idxs, test_doc_labels)

logger.info("Docs remaining -> train: %d dev: %d test: %d",
            len(train_doc_idxs), len(dev_doc_idxs), len(test_doc_idxs))


2026-08-26 19:35:31,226 [INFO] Unique sentence chunks -> train: 1414 dev: 283 test: 273
2026-08-26 19:35:32,122 [INFO] Docs remaining -> train: 300 dev: 60 test: 60


In [15]:
# SpidersReader produces bag-of-words spider diagrams without grammar cups (n . n.r -> I).
# Therefore, RemoveCupsRewriter is redundant and skipped here to avoid unnecessary tree traversals.
# (If using BobcatParser with CCG grammar in the future, RemoveCupsRewriter is used instead).
logger.info("Using spiders_reader diagrams directly (no cups to remove). Diagrams ready: train=%d, dev=%d, test=%d",
            len(train_diagrams), len(dev_diagrams), len(test_diagrams))


2026-08-26 19:35:34,507 [INFO] Using spiders_reader diagrams directly (no cups to remove). Diagrams ready: train=1414, dev=283, test=273


In [16]:
from tqdm.auto import tqdm
from lambeq import IQPAnsatz, AtomicType, PennyLaneModel

ansatz = IQPAnsatz(
    {AtomicType.NOUN: 1, AtomicType.SENTENCE: 1},
    n_layers=N_LAYERS,
    n_single_qubit_params=N_SINGLE_QUBIT_PARAMS,
)

logger.info("Transforming diagrams to quantum circuits...")
train_circuits = [ansatz(d) for d in tqdm(train_diagrams, desc="Building Train Circuits")]
dev_circuits = [ansatz(d) for d in tqdm(dev_diagrams, desc="Building Dev Circuits")]
test_circuits = [ansatz(d) for d in tqdm(test_diagrams, desc="Building Test Circuits")]
logger.info("Circuits built -> train: %d dev: %d test: %d", len(train_circuits), len(dev_circuits), len(test_circuits))


2026-08-26 19:35:35,835 [INFO] Transforming diagrams to quantum circuits...


Building Train Circuits:   0%|          | 0/1414 [00:00<?, ?it/s]

Building Dev Circuits:   0%|          | 0/283 [00:00<?, ?it/s]

Building Test Circuits:   0%|          | 0/273 [00:00<?, ?it/s]

2026-08-26 19:36:05,269 [INFO] Circuits built -> train: 1414 dev: 283 test: 273


## 7. Model, document-level aggregation, training loop

All trainable parameters live in the quantum circuits (`PennyLaneModel`). Document-level
prediction is a parameter-free mean over that document's sentence-circuit outputs -- this
keeps the model purely QNLP, with no hidden classical classifier.


In [17]:
all_circuits = train_circuits + dev_circuits + test_circuits
model = PennyLaneModel.from_diagrams(all_circuits, probabilities=True, normalize=True)
model.initialise_weights()
model = model.double()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
logger.info("Total trainable (quantum) parameters: %d", sum(p.numel() for p in model.parameters()))


def doc_batch_predictions(doc_idx_batch, circuits):
    flat_indices = [i for doc in doc_idx_batch for i in doc]
    if not flat_indices:
        return torch.empty((0, 2), dtype=torch.float64)
    flat_circuits = [circuits[i] for i in flat_indices]
    outputs = model(flat_circuits)
    doc_preds, pos = [], 0
    for doc in doc_idx_batch:
        n = len(doc)
        if n == 0:
            doc_preds.append(torch.tensor([0.5, 0.5], dtype=outputs.dtype, device=outputs.device))
        else:
            doc_preds.append(outputs[pos:pos + n].mean(dim=0))
            pos += n
    return torch.stack(doc_preds)


def accuracy(doc_idxs, circuits, labels):
    preds = doc_batch_predictions(doc_idxs, circuits)[:, 1]
    labels_tensor = torch.tensor(labels, dtype=preds.dtype, device=preds.device)
    return (torch.round(preds) == labels_tensor).sum().item() / max(len(labels), 1)


2026-08-26 19:39:10,488 [INFO] Total trainable (quantum) parameters: 84108


In [18]:
# Device setup: PennyLane default.qubit simulator executes quantum circuits on CPU.
# If using a GPU-accelerated backend (e.g. PennyLane Lightning GPU), device can be cuda.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Using device: %s", device)
try:
    model = model.to(device)
except Exception as e:
    logger.warning("Model could not be moved to %s (%s), keeping on default device", device, e)


2026-08-26 19:39:10,497 [INFO] Using device: cpu


In [19]:
import random as _random
import time

best = {'acc': 0.0, 'epoch': 0}
train_order = list(range(len(train_doc_idxs)))

training_start = time.time()
for epoch in range(EPOCHS):
    epoch_start = time.time()
    _random.shuffle(train_order)
    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, len(train_order), DOC_BATCH_SIZE):
        batch_positions = train_order[start:start + DOC_BATCH_SIZE]
        doc_idx_batch = [train_doc_idxs[i] for i in batch_positions]
        label_batch = [train_doc_labels[i] for i in batch_positions]

        optimizer.zero_grad()
        preds = doc_batch_predictions(doc_idx_batch, train_circuits)[:, 1]
        # Match labels device & dtype dynamically to model prediction tensor (prevents device mismatch)
        labels_t = torch.tensor(label_batch, dtype=preds.dtype, device=preds.device)
        loss = torch.nn.functional.binary_cross_entropy(preds, labels_t)
        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()
        n_batches += 1

    epoch_time = time.time() - epoch_start
    avg_loss = epoch_loss / max(n_batches, 1)
    if epoch == 0:
        logger.info("First epoch took %.1fs -- extrapolate: %d epochs ~= %.1f min total",
                     epoch_time, EPOCHS, epoch_time * EPOCHS / 60)

    if epoch % EVAL_EVERY == 0:
        dev_acc = accuracy(dev_doc_idxs, dev_circuits, dev_doc_labels)
        logger.info("Epoch %d | train_loss=%.4f | dev_acc=%.4f | epoch_time=%.1fs",
                     epoch, avg_loss, dev_acc, epoch_time)
        if dev_acc > best['acc']:
            best['acc'] = dev_acc
            best['epoch'] = epoch
            model.save(MODEL_CHECKPOINT_PATH)
            logger.info("New best dev_acc=%.4f -- checkpoint saved", dev_acc)

logger.info("Training complete in %.1f min. Best dev_acc=%.4f (epoch %d)",
            (time.time() - training_start) / 60, best['acc'], best['epoch'])

if best['acc'] > 0:
    try:
        model.load(MODEL_CHECKPOINT_PATH)
        model = model.double()
    except Exception as e:
        logger.warning("Could not reload best checkpoint: %s", e)


2026-08-26 19:42:06,236 [INFO] First epoch took 175.2s -- extrapolate: 10 epochs ~= 29.2 min total
2026-08-26 19:42:26,275 [INFO] Epoch 0 | train_loss=0.6109 | dev_acc=0.6333 | epoch_time=175.2s
2026-08-26 19:42:48,637 [INFO] New best dev_acc=0.6333 -- checkpoint saved
2026-08-26 19:46:05,466 [INFO] Epoch 1 | train_loss=0.1362 | dev_acc=0.7500 | epoch_time=181.9s
2026-08-26 19:46:33,048 [INFO] New best dev_acc=0.7500 -- checkpoint saved
2026-08-26 19:49:51,780 [INFO] Epoch 2 | train_loss=0.2921 | dev_acc=0.8333 | epoch_time=183.5s
2026-08-26 19:50:18,863 [INFO] New best dev_acc=0.8333 -- checkpoint saved
2026-08-26 19:53:39,594 [INFO] Epoch 3 | train_loss=0.3044 | dev_acc=0.6833 | epoch_time=185.3s
2026-08-26 19:57:06,820 [INFO] Epoch 4 | train_loss=0.2902 | dev_acc=0.6667 | epoch_time=187.3s
2026-08-26 20:00:28,172 [INFO] Epoch 5 | train_loss=0.2509 | dev_acc=0.7667 | epoch_time=186.2s
2026-08-26 20:03:55,557 [INFO] Epoch 6 | train_loss=0.2313 | dev_acc=0.7000 | epoch_time=186.7s
2026

## 8. Evaluation (metrics only -- no classical model involved)

In [20]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

test_preds = doc_batch_predictions(test_doc_idxs, test_circuits)[:, 1]
test_pred_labels = torch.round(test_preds).long().tolist()

acc = accuracy_score(test_doc_labels, test_pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_doc_labels, test_pred_labels, average="binary", zero_division=0
)
cm = confusion_matrix(test_doc_labels, test_pred_labels)

logger.info("Test accuracy:  %.4f", acc)
logger.info("Test precision: %.4f", precision)
logger.info("Test recall:    %.4f", recall)
logger.info("Test F1:        %.4f", f1)
logger.info("Confusion matrix [[TN, FP], [FN, TP]]:\n%s", cm)


2026-08-26 20:15:20,515 [INFO] Test accuracy:  0.7167
2026-08-26 20:15:20,516 [INFO] Test precision: 0.6667
2026-08-26 20:15:20,517 [INFO] Test recall:    0.8667
2026-08-26 20:15:20,518 [INFO] Test F1:        0.7536
2026-08-26 20:15:20,519 [INFO] Confusion matrix [[TN, FP], [FN, TP]]:
[[17 13]
 [ 4 26]]


## 9. Save final model

In [21]:
model.save(MODEL_FINAL_PATH)
logger.info("Model saved to %s", MODEL_FINAL_PATH)


2026-08-26 20:15:41,934 [INFO] Model saved to qnlp_fake_news_final.lt


## 10. Inference function

This is what a serving layer (e.g. the FastAPI example below) would call for a single new
article at request time.


In [25]:
def predict_fake_or_real(text: str) -> dict:
    """Run a single new article through the trained QNLP pipeline."""

    chunks = doc_to_sentence_chunks(text)

    if not chunks:
        return {
            "label": None,
            "confidence": None,
            "p_real": None,
            "chunks_used": 0,
            "error": "no usable sentence chunks extracted from input",
        }

    diagrams = []
    failed_chunks = []

    # IMPORTANT:
    # Training uses parser.sentence2diagram(), so inference must use
    # the same operation.
    for chunk in chunks:
        try:
            diagram = parser.sentence2diagram(chunk)

            if diagram is not None:
                diagrams.append(diagram)
            else:
                failed_chunks.append(chunk)

        except Exception as e:
            failed_chunks.append(chunk)
            logger.warning(
                "Failed to parse sentence: %r | %s",
                chunk,
                e,
            )

    if not diagrams:
        return {
            "label": None,
            "confidence": None,
            "p_real": None,
            "chunks_used": 0,
            "error": "all sentence chunks failed to parse",
        }

    # Same ansatz used during training
    circuits = [ansatz(d) for d in diagrams]

    # Same model used during training
    outputs = model(circuits)

    # outputs shape: [number_of_sentences, 2]
    # class 0 = fake
    # class 1 = real
    doc_output = outputs.mean(dim=0)

    p_fake = doc_output[0].item()
    p_real = doc_output[1].item()

    if p_real >= 0.5:
        label = "real"
        confidence = p_real
    else:
        label = "fake"
        confidence = p_fake

    return {
        "label": label,
        "confidence": confidence,
        "p_fake": p_fake,
        "p_real": p_real,
        "chunks_used": len(diagrams),
        "chunks_failed": len(failed_chunks),
    }

In [26]:
example = (
    test_texts[0]
    if test_texts
    else "Reuters reported that officials confirmed the policy today."
)

result = predict_fake_or_real(example)

logger.info("Sample inference: %s", result)
print(result)

2026-08-26 20:23:41,636 [INFO] Sample inference: {'label': 'real', 'confidence': 0.9998987752081874, 'p_fake': 0.00010122479181266262, 'p_real': 0.9998987752081874, 'chunks_used': 3, 'chunks_failed': 0}


{'label': 'real', 'confidence': 0.9998987752081874, 'p_fake': 0.00010122479181266262, 'p_real': 0.9998987752081874, 'chunks_used': 3, 'chunks_failed': 0}


## 11. Appendix: production serving layer (FastAPI)

Not run in this notebook -- save as `serve.py` and run separately (`uvicorn serve:app`)
once you have a trained `qnlp_fake_news_final.lt` checkpoint. This is the piece that turns
the trained model into something an application can actually call over HTTP.

```python
# serve.py
from fastapi import FastAPI
from pydantic import BaseModel
from lambeq import spiders_reader, RemoveCupsRewriter, IQPAnsatz, AtomicType, PennyLaneModel

# -- must match training config exactly --
N_LAYERS = 2
N_SINGLE_QUBIT_PARAMS = 3
MAX_WORDS_PER_SENTENCE = 12
MAX_SENTENCES_PER_DOC = 20

parser = spiders_reader
remove_cups = RemoveCupsRewriter()
ansatz = IQPAnsatz({AtomicType.NOUN: 1, AtomicType.SENTENCE: 1},
                    n_layers=N_LAYERS, n_single_qubit_params=N_SINGLE_QUBIT_PARAMS)

model = PennyLaneModel.from_checkpoint("qnlp_fake_news_final.lt")
model = model.double()

app = FastAPI(title="QNLP Fake News Classifier")

class ArticleRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(req: ArticleRequest):
    # ... same doc_to_sentence_chunks / predict_fake_or_real logic as the notebook ...
    return predict_fake_or_real(req.text)
```
